# E1 — CIFAR-100 Table 1 comparison

Runs Euclidean NCM, FeCAM shared covariance, and FeCAM classwise covariance with one supplied seed-1 backbone checkpoint. The notebook does not train a backbone. Attach the checkpoint as a Kaggle Dataset, enable Internet, and select a GPU.

Before running: push `feat/e1-covariance-table1` to GitHub; attach the verified `base.pt` checkpoint; set `CHECKPOINT` below to its Kaggle input path. The run summaries enforce identical checkpoint hashes and class order. W&B upload is optional and uses a Kaggle Secret named `WANDB_API_KEY`.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/duongthienphu/fecam-study.git'
REPO_BRANCH = 'feat/e1-covariance-table1'
REPO_DIR = Path('/kaggle/working/fecam-study')
if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', 'PyYAML', 'matplotlib', 'wandb'], check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
# Update this path to the attached seed-1 base.pt checkpoint.
CHECKPOINT = Path('/kaggle/input/e1-seed1-checkpoint/base.pt')
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Attach the verified base.pt Kaggle Dataset and update CHECKPOINT: {CHECKPOINT}')
if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook Settings')
OUTPUT_ROOT = Path('/kaggle/working/e1-seed1')
CONFIGS = {
    'ncm': 'configs/experiments/e1_ncm_euclidean.yaml',
    'fecam_shared': 'configs/experiments/e1_fecam_shared.yaml',
    'fecam_classwise': 'configs/experiments/e1_fecam_classwise.yaml',
}
for name, config in CONFIGS.items():
    output = OUTPUT_ROOT / name
    command = [sys.executable, 'scripts/run_experiment.py', 'evaluate', '--config', config, '--checkpoint', str(CHECKPOINT), '--device', 'cuda:0', '--output', str(output)]
    print('Running:', name, flush=True)
    subprocess.run(command, check=True)
print('All E1 evaluations completed:', OUTPUT_ROOT)

In [ ]:
summaries = [str(OUTPUT_ROOT / name / 'summary.json') for name in CONFIGS]
subprocess.run([sys.executable, 'scripts/report_e1.py', *summaries], check=True)

# Optional W&B upload. Add WANDB_API_KEY as a Kaggle Secret to enable it.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    print('WANDB_API_KEY secret unavailable; skipping W&B upload.')
else:
    subprocess.run([sys.executable, 'scripts/upload_e1_to_wandb.py', *summaries, '--plot', 'results/e1/e1_seen_accuracy.png'], check=True)

import shutil
shutil.make_archive('/kaggle/working/e1-seed1-results', 'zip', root_dir='/kaggle/working', base_dir='e1-seed1')
shutil.make_archive('/kaggle/working/e1-reports', 'zip', root_dir='results', base_dir='e1')
print('Download e1-seed1-results.zip and e1-reports.zip from Kaggle Output.')